### DRMcv Function

#### Cross-Validated OLS for HIV Drug Resistance Mutations.

This is adapted from the R script found here:
https://hivdb.stanford.edu/pages/genopheno.dataset.html
Original script's author: Haley Hedlin (Stanford).

In [92]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, RepeatedKFold

Inputs:

In [93]:
name = "viridian_blue"

In [94]:
dataset = "PI"        # "PI", "NRTI", or "NNRTI"
drug = "LPV"
min_muts = 10
nfold = 5
nrep = 10
confusion = True
lars = True

muts_in = [
    "47A", "84A", "50V", "76V", "82A", "82F", "84V",
    "84C", "82S", "82T", "82M", "32I", "47V", "54M", "54L",
    "54V", "90M", "54A", "54S", "54T", "46I", "46L", "48V",
    "48M", "24I", "82C", "33F", "10F", "73S", "73T", "73C",
    "73A", "11I", "11L", "89V", "20T", "53L", "88S", "50L",
    "24F", "30N", "43T", "46V", "58E", "83D", "88T", "85V",
]

Helper function(s):

In [95]:
def convert_muts(muts_in):
    """Convert insertion/deletion suffixes ('ins'/'del') to '#'/'~'."""
    result = list(muts_in)
    for i, m in enumerate(result):
        if len(m) == 5:
            pos = m[:2]
            if m[2:5] == "ins":
                result[i] = pos + "#"
            elif m[2:5] == "del":
                result[i] = pos + "~"
        elif len(m) == 6:
            pos = m[:3]
            if m[3:6] == "ins":
                result[i] = pos + "#"
            elif m[3:6] == "del":
                result[i] = pos + "~"
    return result


def check_muts(muts_in):
    """Validate that mutation codes are well-formed."""
    valid_last = set("ABCDEFGHIJKLMNOPQRSTUVWXYZ#~")
    for m in muts_in:
        if not (3 <= len(m) <= 4):
            raise ValueError(
                'All entries in muts_in should be between 3 and 4 characters long.'
            )
        if len(m) == 3:
            if m[2].upper() not in valid_last:
                raise ValueError(
                    'Last character must be a letter, #, or ~.'
                )
            try:
                int(m[:2])
            except ValueError:
                raise ValueError('First two characters must be digits.')
        elif len(m) == 4:
            if m[3].upper() not in valid_last:
                raise ValueError(
                    'Last character must be a letter, #, or ~.'
                )
            try:
                int(m[:3])
            except ValueError:
                raise ValueError('First three characters must be digits.')


def build_x(posu, mut, ps):
    """Build the binary design matrix from position data."""
    n = len(posu)
    k = len(mut)
    X = np.full((n, k), np.nan)

    for p in set(ps):
        col_data = posu.iloc[:, p].astype(str)
        p1 = col_data.str[0]
        p2 = col_data.str[1]
        for ind in [i for i, x in enumerate(ps) if x == p]:
            X[:, ind] = ((p1 == mut[ind]) | (p2 == mut[ind])).astype(float)

    col_names = [f"{ps[i]}{mut[i]}" for i in range(k)]
    return pd.DataFrame(X, columns=col_names, index=posu.index)

Validation:

In [96]:
if drug == "3TC":
    drug = "X3TC"

muts_in_conv = convert_muts(muts_in)
check_muts(muts_in_conv)

# Extract amino acid letter and numeric position from each mutation code
mut = []
ps = []
for m in muts_in:
    if len(m) == 3:
        mut.append(m[2].upper())
        ps.append(int(m[:2]))
    else:
        mut.append(m[3].upper())
        ps.append(int(m[:3]))

print(f"{len(mut)} mutations to include in model")

47 mutations to include in model


Load data:

In [97]:
urls = {
    "PI":    "http://hivdb.stanford.edu/download/GenoPhenoDatasets/PI_DataSet.txt",
    "NRTI":  "http://hivdb.stanford.edu/download/GenoPhenoDatasets/NRTI_DataSet.txt",
    "NNRTI": "http://hivdb.stanford.edu/download/GenoPhenoDatasets/NNRTI_DataSet.txt",
}

pos_slices = {
    "PI":    slice(9, 108),    # columns 10:108 (0-indexed: 9:108)
    "NRTI":  slice(7, 247),    # columns 8:247
    "NNRTI": slice(5, 245),    # columns 6:245
}

if dataset not in urls:
    raise ValueError('dataset must be "PI", "NRTI", or "NNRTI".')

dat = pd.read_csv(urls[dataset], sep="\t", comment="@", na_values=".")
posu = dat.iloc[:, pos_slices[dataset]]

print(f"Loaded {len(dat)} rows, {posu.shape[1]} position columns")

Loaded 2171 rows, 99 position columns


Build matrix:

In [98]:
X = build_x(posu, mut, ps)

Y = pd.to_numeric(dat[drug], errors="coerce")
Ylog10 = np.log10(Y)

df_log = pd.concat([Ylog10.rename("Y"), X], axis=1)

# Complete cases only
df_log_cc = df_log.dropna().copy()
print(f"{len(df_log_cc)} complete cases out of {len(df_log)}")

1807 complete cases out of 2171


Remove rare mutations:

In [99]:
feature_cols = [c for c in df_log_cc.columns if c != "Y"]
col_sums = df_log_cc[feature_cols].sum()
rare = col_sums[col_sums < min_muts].index.tolist()

if rare:
    for r in rare:
        print(f"{r} excluded (appears in fewer than {min_muts} sequences)")
    df_log_cc = df_log_cc.drop(columns=rare)
    feature_cols = [c for c in feature_cols if c not in rare]

print(f"{len(feature_cols)} mutations retained in model")

47A excluded (appears in fewer than 10 sequences)
84A excluded (appears in fewer than 10 sequences)
50V excluded (appears in fewer than 10 sequences)
82A excluded (appears in fewer than 10 sequences)
82F excluded (appears in fewer than 10 sequences)
84C excluded (appears in fewer than 10 sequences)
82S excluded (appears in fewer than 10 sequences)
82T excluded (appears in fewer than 10 sequences)
82M excluded (appears in fewer than 10 sequences)
54M excluded (appears in fewer than 10 sequences)
54L excluded (appears in fewer than 10 sequences)
54V excluded (appears in fewer than 10 sequences)
90M excluded (appears in fewer than 10 sequences)
54A excluded (appears in fewer than 10 sequences)
54S excluded (appears in fewer than 10 sequences)
54T excluded (appears in fewer than 10 sequences)
46I excluded (appears in fewer than 10 sequences)
46L excluded (appears in fewer than 10 sequences)
48V excluded (appears in fewer than 10 sequences)
48M excluded (appears in fewer than 10 sequences)


Fit OLS and run CV:

In [100]:
Xmat = df_log_cc[feature_cols]
Yvec = df_log_cc["Y"]

# --- Full OLS fit (statsmodels gives coefficients + SEs like R's lm) ---
Xmat_const = sm.add_constant(Xmat)
ols_fit = sm.OLS(Yvec, Xmat_const).fit()

ols_coefs = pd.DataFrame({
    "Estimate": ols_fit.params,
    "Std. Error": ols_fit.bse,
})
print(ols_coefs.to_string())

       Estimate  Std. Error
const  0.567942    0.021709
76V   -0.239219    0.093430
84V    0.409200    0.085347
32I    0.523706    0.103771
47V    0.936776    0.096936
73S    0.376692    0.085733
73T   -0.343495    0.204931
73A    0.389220    0.186533
11I    0.084151    0.188822
53L    0.599706    0.076706
46V    1.037745    0.076999


In [101]:
# --- Cross-validation (repeated k-fold, matching caret's train()) ---
lr = LinearRegression()
cv = RepeatedKFold(n_splits=nfold, n_repeats=nrep, random_state=42)

# neg_mean_squared_error → negate to get MSE
mse_scores = -cross_val_score(lr, Xmat.values, Yvec.values, cv=cv,
                              scoring="neg_mean_squared_error")

print(f"Mean CV MSE: {mse_scores.mean():.4f}  (SD: {mse_scores.std():.4f})")

Mean CV MSE: 0.6350  (SD: 0.0323)


Confusion matrix:

In [102]:
if confusion:
    cutoff_map = {
        "FPV":  (3, 15),   "ATV":  (3, 15),   "IDV":  (3, 15),
        "LPV":  (9, 55),   "NFV":  (3, 6),    "SQV":  (3, 15),
        "TPV":  (2, 8),    "DRV":  (10, 90),
        "X3TC": (5, 25),   "ABC":  (2, 6),    "AZT":  (3, 15),
        "D4T":  (1.5, 3),  "DDI":  (1.5, 3),  "TDF":  (1.5, 3),
        "EFV":  (3, 10),   "NVP":  (3, 10),   "ETR":  (3, 10),
        "RPV":  (3, 10),
    }

    lower, upper = cutoff_map[drug]
    bins = [0, lower, upper, np.inf]
    labels_cat = ["susceptible", "intermediate-level resistant", "high-level resistant"]

    predicted = pd.cut(10 ** ols_fit.predict(Xmat_const), bins=bins, labels=labels_cat)
    actual = pd.cut(10 ** Yvec, bins=bins, labels=labels_cat)

    conf_tab = pd.crosstab(predicted, actual, dropna=False)
    print(conf_tab)
else:
    print("Confusion matrix not requested (set confusion = True to enable).")

Y                             susceptible  intermediate-level resistant  \
row_0                                                                     
susceptible                           936                           246   
intermediate-level resistant           70                            95   
high-level resistant                    7                            13   

Y                             high-level resistant  
row_0                                               
susceptible                                    228  
intermediate-level resistant                   159  
high-level resistant                            53  


LARS and Elastic net via `glmnet`:

In [103]:
if lars:
    from sklearn.linear_model import LassoCV

    # LassoCV is the closest sklearn analogue to glmnet's cv.glmnet with alpha=1
    lasso_cv = LassoCV(cv=5, random_state=42)
    lasso_cv.fit(Xmat.values, Yvec.values)

    lars_coefs = pd.Series(lasso_cv.coef_, index=feature_cols, name="LARS_coef")
    lars_coefs.loc["(Intercept)"] = lasso_cv.intercept_
    print(lars_coefs.to_string())
else:
    print("LARS not requested (set lars = True to enable).")

76V           -0.219936
84V            0.390959
32I            0.498043
47V            0.924417
73S            0.354697
73T           -0.219189
73A            0.288184
11I            0.011181
53L            0.589551
46V            1.028479
(Intercept)    0.572566


Save the outputs:

In [104]:
ols_coefs.to_csv(f'geno-pheno-outputs/OLScoefs/{name}.txt', sep=" ")
pd.DataFrame({"CV_MSE": mse_scores}).to_csv(f'geno-pheno-outputs/CVmse/{name}.txt', sep=" ")

if confusion:
    conf_tab.to_csv(f'geno-pheno-outputs/Confusion/{name}.txt', sep=" ")

if lars:
    lars_coefs.to_csv(f'geno-pheno-outputs/LARScoef/{name}.txt', sep=" ")

print("Files written.")

Files written.
